# Fine-tune YOLO26 balon - Colab T4 (contingencia P1.3b)

Via de CONTINGENCIA. La principal es ROCm local (RX 9070 XT, ver `docs/plan-finetune-balon.md`).
Usar esto solo si el train local falla.

Runtime -> Cambiar tipo de entorno -> GPU (T4). Mismos params que `scripts/train_ball.py`.

Pasos: 1) verificar GPU  2) instalar  3) subir dataset zip  4) train  5) bajar best.pt

In [ ]:
!nvidia-smi

In [ ]:
!pip -q install ultralytics

Sube el zip del dataset de balon (`volleyball.v1i.yolov8`). Exportar de Roboflow como YOLOv8 TXT, o comprimir la carpeta local.

In [ ]:
import glob, zipfile, os
from google.colab import files
files.upload()  # sube volleyball.v1i.yolov8.zip
z = glob.glob('*.zip')[0]
with zipfile.ZipFile(z) as f:
    f.extractall('dataset')
print('extraido en dataset/:', os.listdir('dataset'))

In [ ]:
# Reconstruye data.yaml robusto (ignora el '../' roto del export Roboflow)
from pathlib import Path
import yaml
root = next(Path('dataset').rglob('data.yaml')).parent
src = yaml.safe_load((root / 'data.yaml').read_text())
cfg = {'path': str(root.resolve()), 'nc': src.get('nc', 1), 'names': src.get('names', ['balon'])}
for split, key in (('train', 'train'), ('valid', 'val'), ('test', 'test')):
    if (root / split / 'images').is_dir():
        cfg[key] = f'{split}/images'
Path('ball.yaml').write_text(yaml.safe_dump(cfg, allow_unicode=True))
print(cfg)

In [ ]:
# Mismos params que scripts/train_ball.py (T4 16GB -> batch 8)
from ultralytics import YOLO
m = YOLO('yolo26n.pt')  # ultralytics lo descarga solo
m.train(data='ball.yaml', epochs=80, patience=20, imgsz=1280, batch=8,
        device=0, workers=2, cache=False, project='runs', name='ball_ft')

In [ ]:
# Baja best.pt -> copiar a models/ del repo y correr validate_ball
from google.colab import files
files.download('runs/ball_ft/weights/best.pt')